In [1]:
# Fix 1: Make sure to use consistent imports (either PyQt5 or PyQt6)
from PyQt6.QtWidgets import QApplication, QMainWindow, QWidget, QGridLayout, QLabel, QPushButton, QVBoxLayout, QFileDialog, QHBoxLayout, QFrame, QProgressDialog, QSizePolicy, QDialog, QDialogButtonBox, QSlider
from PyQt6.QtGui import QMovie, QPixmap, QPainter, QPen, QColor, QImage
from PyQt6.QtCore import Qt, QSize, pyqtSignal

import numpy as np
import matplotlib.pyplot as plt
import sys
from skimage import io, color, transform, img_as_ubyte
from skimage.metrics import structural_similarity as ssim

def gaussian_rbf(r):
    return np.exp(-r**2)

def kernel_function(x, z, gamma_x, gamma_z, s1, s2, exponent, phi_func):
    spatial_dist = np.linalg.norm(x[:, None, :] - z[None, :, :], axis=-1)
    spatial_kernel = phi_func(spatial_dist / s1)

    grayscale_diff = np.abs(gamma_x[:, None] - gamma_z[None, :]) ** exponent
    grayscale_kernel = phi_func(grayscale_diff / s2)

    return spatial_kernel * grayscale_kernel

# Fix 2: Update FloatSlider to use PyQt6 instead of PyQt5
class FloatSlider(QSlider):
    # Custom signal that emits float values
    floatValueChanged = pyqtSignal(float)
    
    def __init__(self, orientation, parent=None):
        super().__init__(orientation, parent)
        self.min_float = 0.0
        self.max_float = 1.0
        self.scale_factor = 100000  # High precision for small values
        self.valueChanged.connect(self._emit_float_value)
        
    def set_float_range(self, min_val, max_val):
        self.min_float = min_val
        self.max_float = max_val
        # Reset integer range based on new float range
        self.setMinimum(0)
        self.setMaximum(int((max_val - min_val) * self.scale_factor))
        
    def _emit_float_value(self):
        # Convert the integer value to a float in the desired range
        float_value = self.min_float + (self.value() / self.scale_factor)
        self.floatValueChanged.emit(float_value)
        
    def set_float_value(self, value):
        # Convert a float value to the integer range of the slider
        if value < self.min_float:
            value = self.min_float
        if value > self.max_float:
            value = self.max_float
            
        integer_value = int((value - self.min_float) * self.scale_factor)
        self.setValue(integer_value)
        
    def float_value(self):
        # Return the current value as a float
        return self.min_float + (self.value() / self.scale_factor)

class MainWindow(QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Image Colourisation")
        self.setMinimumSize(QSize(800, 600))

        self.load_button = QPushButton("Load Image")
        self.load_button.clicked.connect(self.load_image)
        # self.load_button.setSizePolicy(QSizePolicy.Policy.Expanding, QSizePolicy.Policy.Expanding)  # Expands

        self.title_1 = QLabel("Original Image")
        self.title_1.setAlignment(Qt.AlignmentFlag.AlignTop | Qt.AlignmentFlag.AlignHCenter)
        self.title_1.setStyleSheet("font-size: 12px; font-weight: bold; padding-top: 20px; padding-bottom: 20px;")
        # self.title_1.setSizePolicy(QSizePolicy.Policy.Preferred, QSizePolicy.Policy.Fixed)  # Fix height

        self.layout_1 = QVBoxLayout()
        self.layout_1.addWidget(self.title_1)
        self.layout_1.addStretch(1)
        self.layout_1.addWidget(self.load_button, alignment=Qt.AlignmentFlag.AlignCenter)
        self.layout_1.addStretch(1)

        self.grayscale_button = QPushButton("Grayscale")
        self.grayscale_button.setEnabled(False)
        self.grayscale_button.clicked.connect(self.grayscale)
        self.grayscale_button.setSizePolicy(QSizePolicy.Policy.Expanding, QSizePolicy.Policy.Expanding)  # Expands

        self.title_2 = QLabel("Grayscale Image")
        self.title_2.setAlignment(Qt.AlignmentFlag.AlignTop | Qt.AlignmentFlag.AlignHCenter)
        self.title_2.setStyleSheet("font-size: 12px; font-weight: bold; padding-top: 20px; padding-bottom: 10px;")
        # self.title_2.setSizePolicy(QSizePolicy.Policy.Preferred, QSizePolicy.Policy.Fixed)
        
        self.layout_2 = QVBoxLayout()
        self.layout_2.addWidget(self.title_2, alignment=Qt.AlignmentFlag.AlignCenter)
        self.layout_2.addStretch(1)
        self.layout_2.addWidget(self.grayscale_button, alignment=Qt.AlignmentFlag.AlignCenter)
        self.layout_2.addStretch(1)
        
        self.random_sample_button = QPushButton("Random Sample")
        self.random_sample_button.setEnabled(False)
        self.random_sample_button.clicked.connect(self.random_sample)
        self.random_sample_button.setSizePolicy(QSizePolicy.Policy.Expanding, QSizePolicy.Policy.Expanding)  # Expands

        self.uniform_sample_button = QPushButton("Uniform Sample")
        self.uniform_sample_button.setEnabled(False)
        self.uniform_sample_button.clicked.connect(self.uniform_sample)

        # Added this to add select sample button
        self.select_sample_button = QPushButton("Select Sample")
        self.select_sample_button.setEnabled(False)
        self.select_sample_button.clicked.connect(self.select_sample)

        # Here I am adding a horizontal slider

        self.current_sigma_1 = 1000.0
        self.current_sigma_2 = 0.55
        self.current_p = 0.5
        self.current_delta = 1e-5
        self.current_colour_fraction = 0.01

        self.add_horizontal_slider = QSlider(Qt.Orientation.Horizontal)
        self.add_horizontal_slider.setFixedWidth(200)
        self.add_horizontal_slider.setMinimum(-10)
        self.add_horizontal_slider.setMaximum(10)
        self.add_horizontal_slider.setSingleStep(1)
        self.add_horizontal_slider.setValue(0) # I think I can do 1-10 in order to make it more user-friendly

        self.add_horizontal_slider_sigma_2 = QSlider(Qt.Orientation.Horizontal)
        self.add_horizontal_slider_sigma_2.setFixedWidth(200)
        self.add_horizontal_slider_sigma_2.setMinimum(-10)
        self.add_horizontal_slider_sigma_2.setMaximum(10)
        self.add_horizontal_slider_sigma_2.setSingleStep(1)
        self.add_horizontal_slider_sigma_2.setValue(0)

        self.add_horizontal_slider_p = QSlider(Qt.Orientation.Horizontal)
        self.add_horizontal_slider_p.setFixedWidth(200)
        self.add_horizontal_slider_p.setMinimum(-10)
        self.add_horizontal_slider_p.setMaximum(10)
        self.add_horizontal_slider_p.setSingleStep(1)
        self.add_horizontal_slider_p.setValue(0)

        # Replace your current slider code with this
        self.add_horizontal_slider_delta = FloatSlider(Qt.Orientation.Horizontal)
        self.add_horizontal_slider_delta.setFixedWidth(200)
        self.add_horizontal_slider_delta.set_float_range(0, 10e-5)  # Set the float range
        self.add_horizontal_slider_delta.set_float_value(0)  # Set initial value as float

        # Connect to your handler
        self.add_horizontal_slider_delta.floatValueChanged.connect(self.on_delta_changed)

        # Colour fraction slider
        self.add_horizontal_slider_colour_fraction = FloatSlider(Qt.Orientation.Horizontal)
        self.add_horizontal_slider_colour_fraction.setFixedWidth(200)
        self.add_horizontal_slider_colour_fraction.set_float_range(0.001, 0.05)  # Set the float range
        self.add_horizontal_slider_colour_fraction.set_float_value(0.01)  # Set initial value as float

        self.add_horizontal_slider_colour_fraction.floatValueChanged.connect(self.on_colour_fraction_changed)
        self.add_horizontal_slider_colour_fraction.sliderMoved.connect(self.slider_position)
        self.add_horizontal_slider_colour_fraction.sliderPressed.connect(self.slider_pressed)
        self.add_horizontal_slider_colour_fraction.sliderReleased.connect(self.slider_released)

        

        # For sigma_1:
        self.add_horizontal_slider.valueChanged.connect(self.slider_sigma_1_changed)
        self.add_horizontal_slider.sliderMoved.connect(self.slider_position)
        self.add_horizontal_slider.sliderPressed.connect(self.slider_pressed)
        self.add_horizontal_slider.sliderReleased.connect(self.slider_released)

        # For sigma_2:
        self.add_horizontal_slider_sigma_2.valueChanged.connect(self.slider_sigma_2_changed)
        self.add_horizontal_slider_sigma_2.sliderMoved.connect(self.slider_position)
        self.add_horizontal_slider_sigma_2.sliderPressed.connect(self.slider_pressed)
        self.add_horizontal_slider_sigma_2.sliderReleased.connect(self.slider_released)

        # For p:
        self.add_horizontal_slider_p.valueChanged.connect(self.slider_p_changed)
        self.add_horizontal_slider_p.sliderMoved.connect(self.slider_position)
        self.add_horizontal_slider_p.sliderPressed.connect(self.slider_pressed)
        self.add_horizontal_slider_p.sliderReleased.connect(self.slider_released)

        # For delta:
        # Fixed: Using floatValueChanged instead of valueChanged to match our custom signal
        self.add_horizontal_slider_delta.floatValueChanged.connect(self.on_delta_changed)
        self.add_horizontal_slider_delta.sliderMoved.connect(self.slider_position)
        self.add_horizontal_slider_delta.sliderPressed.connect(self.slider_pressed)
        self.add_horizontal_slider_delta.sliderReleased.connect(self.slider_released)

        self.title_3 = QLabel("Sampled Image")
        self.title_3.setAlignment(Qt.AlignmentFlag.AlignTop | Qt.AlignmentFlag.AlignHCenter)
        self.title_3.setStyleSheet("font-size: 12px; font-weight: bold; padding-top: 20px; padding-bottom: 10px;")
        # self.title_3.setSizePolicy(QSizePolicy.Policy.Preferred, QSizePolicy.Policy.Fixed)

        self.layout_3 = QVBoxLayout()
        self.layout_3.addWidget(self.title_3, alignment=Qt.AlignmentFlag.AlignCenter)
        self.layout_3.addStretch(1)
        self.layout_3.addWidget(self.random_sample_button, alignment=Qt.AlignmentFlag.AlignCenter)
        self.layout_3.addWidget(self.select_sample_button, alignment=Qt.AlignmentFlag.AlignCenter)
        self.layout_3.addStretch(1)
        # self.layout_3.addWidget(self.uniform_sample_button, alignment=Qt.AlignmentFlag.AlignCenter)
        # self.layout_3.addWidget(self.select_sample_button, alignment=Qt.AlignmentFlag.AlignCenter) # Added this 

        self.colourize_button = QPushButton("Colourize")
        self.colourize_button.setEnabled(False)
        self.colourize_button.clicked.connect(self.colourize)
        self.colourize_button.setSizePolicy(QSizePolicy.Policy.Expanding, QSizePolicy.Policy.Expanding)  # Expands

        self.title_4 = QLabel("Recolourized Image")
        self.title_4.setAlignment(Qt.AlignmentFlag.AlignTop | Qt.AlignmentFlag.AlignHCenter)
        self.title_4.setStyleSheet("font-size: 12px; font-weight: bold; padding-top: 20px; padding-bottom: 10px;")
        # self.title_4.setSizePolicy(QSizePolicy.Policy.Preferred, QSizePolicy.Policy.Fixed)

        self.layout_4 = QVBoxLayout()
        self.layout_4.addWidget(self.title_4, alignment=Qt.AlignmentFlag.AlignCenter)
        self.layout_4.addStretch(1)
        self.layout_4.addWidget(self.colourize_button, alignment=Qt.AlignmentFlag.AlignCenter)
        self.layout_4.addStretch(1)

        """
        self.slider_layout = QVBoxLayout()
        self.slider_layout.addStretch(1)
        self.slider_layout.addWidget(self.add_horizontal_slider)   # Adding the slider in its own column and positioning it agreeably
        self.slider_layout.addStretch(1)
        """

        self.colour_fraction_label = QLabel("Colour fraction = 0.01")
        self.colour_fraction_label.setStyleSheet("font-size: 12px; font-weight: bold; padding-top: 20px; padding-bottom: 20px;")
        self.sigma_label = QLabel("σ₁ = 1000.0")
        self.sigma_label.setStyleSheet("font-size: 12px; font-weight: bold; padding-top: 20px; padding-bottom: 20px;")
        self.sigma_2_label = QLabel("σ₂ = 0.55")
        self.sigma_2_label.setStyleSheet("font-size: 12px; font-weight: bold; padding-top: 20px; padding-bottom: 20px;")
        self.slider_layout = QVBoxLayout()
        self.slider_layout.addWidget(self.colour_fraction_label, alignment=Qt.AlignmentFlag.AlignJustify)
        self.slider_layout.addWidget(self.add_horizontal_slider_colour_fraction)
        self.slider_layout.addWidget(self.sigma_label, alignment=Qt.AlignmentFlag.AlignJustify)
        self.slider_layout.addWidget(self.add_horizontal_slider)
        self.slider_layout.addWidget(self.sigma_2_label, alignment=Qt.AlignmentFlag.AlignJustify)
        self.slider_layout.addWidget(self.add_horizontal_slider_sigma_2)
        """
        self.slider_layout_sigma_2 = QVBoxLayout()
        self.slider_layout_sigma_2.addStretch(1)
        self.slider_layout_sigma_2.addWidget(self.add_horizontal_slider_sigma_2, alignment=Qt.AlignmentFlag.AlignJustify)   # Adding the slider in its own column and positioning it agreeably
        self.slider_layout_sigma_2.addStretch(1)
        """
        """
        self.slider_layout_p = QVBoxLayout()
        self.slider_layout_p.addStretch(1)
        self.slider_layout_p.addWidget(self.add_horizontal_slider_p, alignment=Qt.AlignmentFlag.AlignJustify)   # Adding the slider in its own column and positioning it agreeably
        self.slider_layout_p.addStretch(1)
        """

        self.recolourize_button = QPushButton("Recolourize")
        self.recolourize_button.setEnabled(False)
        self.recolourize_button.clicked.connect(self.colourize)
        self.recolourize_button.setSizePolicy(QSizePolicy.Policy.Expanding, QSizePolicy.Policy.Expanding)

        self.p_label = QLabel("p = 0.5")
        self.p_label.setStyleSheet("font-size: 12px; font-weight: bold; padding-top: 20px; padding-bottom: 20px;")
        self.delta_label = QLabel("δ = 1e-5")
        self.delta_label.setStyleSheet("font-size: 12px; font-weight: bold; padding-top: 20px; padding-bottom: 20px;")
        self.slider_layout_2 = QVBoxLayout()
        self.slider_layout_2.addWidget(self.p_label, alignment=Qt.AlignmentFlag.AlignJustify)
        self.slider_layout_2.addWidget(self.add_horizontal_slider_p, alignment=Qt.AlignmentFlag.AlignJustify)
        self.slider_layout_2.addWidget(self.delta_label, alignment=Qt.AlignmentFlag.AlignJustify)
        self.slider_layout_2.addWidget(self.add_horizontal_slider_delta, alignment=Qt.AlignmentFlag.AlignJustify)
        self.slider_layout_2.addWidget(self.recolourize_button, alignment=Qt.AlignmentFlag.AlignJustify)

        self.main_layout = QGridLayout()
        self.main_layout.addLayout(self.layout_1, 0, 0)
        self.main_layout.addLayout(self.layout_2, 0, 2)
        self.main_layout.addLayout(self.layout_3, 2, 0)
        self.main_layout.addLayout(self.layout_4, 2, 2)
        self.main_layout.addLayout(self.slider_layout, 0, 3)  # row=0
        # self.main_layout.addLayout(self.slider_layout_sigma_2, 1, 4, 1, 1)  # row=1
        self.main_layout.addLayout(self.slider_layout_2, 2, 3)  # row=2
        # self.main_layout.addLayout(self.slider_layout_delta, 3, 4, 1, 1)  # row=3
        # self.main_layout.setColumnStretch(3, 1)
        # self.main_layout.setColumnMinimumWidth(3, 200) 

        # Create horizontal line
        h_line = QFrame()
        h_line.setFrameShape(QFrame.Shape.HLine)
        h_line.setFrameShadow(QFrame.Shadow.Sunken)
        h_line.setStyleSheet("background-color: black; height: 2px;")  # Optional styling

        # Create vertical line 1
        v_line_1 = QFrame()
        v_line_1.setFrameShape(QFrame.Shape.VLine)
        v_line_1.setFrameShadow(QFrame.Shadow.Sunken)
        v_line_1.setStyleSheet("background-color: black; width: 2px;")  # Optional styling

        # Create vertical line 2
        v_line_2 = QFrame()
        v_line_2.setFrameShape(QFrame.Shape.VLine)
        v_line_2.setFrameShadow(QFrame.Shadow.Sunken)
        v_line_2.setStyleSheet("background-color: black; width: 2px;")  # Optional styling

        # Add dividers to the grid
        self.main_layout.addWidget(h_line, 1, 0, 1, 4)  # Row 1, spanning 3 columns
        self.main_layout.addWidget(v_line_1, 0, 1, 4, 1)  # Column 1, spanning 3 rows
        self.main_layout.addWidget(v_line_2, 0, 3, 4, 1)  # Column 1, spanning 3 rows

        self.main_layout.setRowStretch(0, 1)
        self.main_layout.setRowStretch(2, 1)
        self.main_layout.setColumnStretch(3, 1)

        self.main_layout.setColumnStretch(0, 1)
        self.main_layout.setColumnStretch(2, 1)
        self.main_layout.setColumnStretch(3, 1)

        self.main_layout.setContentsMargins(0,0,0,0)
        self.main_layout.setSpacing(0)

        main_widget = QWidget()
        main_widget.setLayout(self.main_layout)
        self.setCentralWidget(main_widget)

    def slider_sigma_1_changed(self, i):
        # i: -10..10
        base = 1000.0
        # Example: ±10% => i/10 => -1..+1 => 0..2000
        multiplier = 1 + (i / 10)
        self.current_sigma_1 = base * multiplier
        self.sigma_label.setText(f"σ₁ = {self.current_sigma_1:.2f}")

    def slider_sigma_2_changed(self, i):
        # i: -10..10
        base = 0.55
        # Example: ±10% => i/10 => -1..+1
        multiplier = 1 + (i / 10)
        self.current_sigma_2 = base * multiplier  # <--- IMPORTANT FIX
        self.sigma_2_label.setText(f"σ₂ = {self.current_sigma_2:.2f}")

    def slider_p_changed(self, i):
        # i: -10..10
        base = 0.5
        # Example: ±10% => i/10 => -1..+1
        multiplier = 1 + (i / 10)
        self.current_p = base * multiplier
        self.p_label.setText(f"p = {self.current_p:.2f}")

    # Fix 4: Added the missing on_delta_changed function
    def on_delta_changed(self, value):
        # Update current delta value with the float value from the slider
        self.current_delta = value
        self.delta_label.setText(f"δ = {self.current_delta:.7f}")

    def on_colour_fraction_changed(self, value):
        self.current_colour_fraction = value
        self.colour_fraction_label.setText(f"Colour fraction = {self.current_colour_fraction:.5f}")

    def slider_position(self, p):
        print("Position", p)

    def slider_pressed(self):
        print("Pressed!")

    def slider_released(self):
        print("Released")    

    def load_image(self):
        self.image_path, _ = QFileDialog.getOpenFileName(self, "Open File", "", "Images (*.png *.jpg *.jpeg *.bmp);;All Files (*)")

        if self.image_path:
            self.image_widget = QLabel()
            pixmap = QPixmap(self.image_path)
            self.image_widget.setPixmap(pixmap)
            self.image_widget.setAlignment(Qt.AlignmentFlag.AlignCenter)
            max_width, max_height = 400, 400
            self.image_widget.setPixmap(pixmap.scaled(max_width, max_height, Qt.AspectRatioMode.KeepAspectRatio, Qt.TransformationMode.SmoothTransformation)) #Rescaled image size to keep GUI of sound proportion
            self.main_layout.addWidget(self.image_widget, 0, 0)
            self.grayscale_button.setEnabled(True)
    
    def grayscale(self):
        print(f"Reading input image: {self.image_path}")
        image = io.imread(self.image_path)
        if image.shape[2] == 4:
            image = image[:, :, :3]  # Strip alpha channel
        self.original_image = image
        #self.original_image = io.imread(self.image_path)

        self.grayscale_path = ".\\greyscale.jpeg" # should be improved
        self.grayscale_image = color.rgb2gray(self.original_image)  # float in [0,1]
        print(f"Saving grayscale image to: {self.grayscale_path}")
        io.imsave(self.grayscale_path, img_as_ubyte(self.grayscale_image))

        self.grayscale_widget = QLabel()
        pixmap = QPixmap(self.grayscale_path)
        self.grayscale_widget.setPixmap(pixmap)
        max_width, max_height = 400, 400
        self.grayscale_widget.setAlignment(Qt.AlignmentFlag.AlignCenter)
        self.grayscale_widget.setPixmap(pixmap.scaled(max_width, max_height, Qt.AspectRatioMode.KeepAspectRatio, Qt.TransformationMode.SmoothTransformation)) # Rescaled Image here also
        self.main_layout.addWidget(self.grayscale_widget, 0, 2)
        self.random_sample_button.setEnabled(True)

        self.select_sample_button.setEnabled(True) # ADDED THIS


    def uniform_sample(self):
        pass

    def random_sample(self):
        colour_fraction = self.current_colour_fraction # fixed for the moment

        self.height, self.width = self.grayscale_image.shape
        total_pixels = self.height * self.width
        num_coloured_pixels = int(total_pixels * colour_fraction)

        selected_indices = np.random.choice(total_pixels, num_coloured_pixels, replace=False)
        self.y_coords, self.x_coords = np.unravel_index(selected_indices, (self.height, self.width))

        self.original_colours = self.original_image.copy()
        self.f = self.original_colours[self.y_coords, self.x_coords, :].astype(float)

        grayscale_uint8 = (self.grayscale_image * 255).astype(np.uint8)
        self.sampled_image = np.stack([grayscale_uint8]*3, axis=-1)
        self.sampled_image[self.y_coords, self.x_coords] = self.original_colours[self.y_coords, self.x_coords]

        self.sampled_path = ".\\sampled.jpeg" # should be improved
        print(f"Saving randomly sampled image to: {self.sampled_path}")
        io.imsave(self.sampled_path, img_as_ubyte(self.sampled_image))

        self.sampled_widget = QLabel()
        pixmap = QPixmap(self.sampled_path)
        self.sampled_widget.setPixmap(pixmap)
        max_width, max_height = 400, 400
        self.sampled_widget.setAlignment(Qt.AlignmentFlag.AlignCenter)
        self.sampled_widget.setPixmap(pixmap.scaled(max_width, max_height, Qt.AspectRatioMode.KeepAspectRatio, Qt.TransformationMode.SmoothTransformation)) #Rescaled Image here also
        self.main_layout.addWidget(self.sampled_widget, 2, 0)
        self.colourize_button.setEnabled(True)

    # Added this method for the sample selection functionality
    def select_sample(self):
        self.sampled_path = "sampled_image.jpeg"
        self.x_coords, self.y_coords = np.array([]), np.array([])
        self.height, self.width = self.grayscale_image.shape
        sample_window = SelectionWindow(self)
        sample_window.exec()

        self.original_colours = self.original_image.copy()
        self.f = self.original_colours[self.y_coords, self.x_coords, :].astype(float)
        print(f"Number of pixels selected: {len(self.x_coords)} \n\
              Fraction of pixels selected: {3*len(self.x_coords)/self.original_colours.size: .2f}")

        self.sampled_widget = QLabel()
        pixmap = QPixmap(self.sampled_path)
        self.sampled_widget.setPixmap(pixmap)
        max_width, max_height = 400, 400
        self.sampled_widget.setAlignment(Qt.AlignmentFlag.AlignCenter)
        self.sampled_widget.setPixmap(pixmap.scaled(max_width, max_height, Qt.AspectRatioMode.KeepAspectRatio, Qt.TransformationMode.SmoothTransformation))
        self.main_layout.addWidget(self.sampled_widget, 2, 0)
        self.colourize_button.setEnabled(True)

        #print(self.x_coords, self.y_coords)

    def colourize(self):
        self.layout_4.removeWidget(self.colourize_button)

        self.spinner_label = QLabel()
        self.spinner_label.setAlignment(Qt.AlignmentFlag.AlignCenter)

        self.spinner = QMovie("loading.gif")  # Provide a valid GIF path
        self.spinner_label.setMovie(self.spinner)

        self.layout_4.addWidget(self.spinner_label, alignment=Qt.AlignmentFlag.AlignCenter)

        self.spinner.start()

        # Fixed for the moment ################################################################ Need to fiddle ########################################################
        sigma_1 = self.current_sigma_1
        sigma_2 = self.current_sigma_2
        p = self.current_p
        delta = self.current_delta

        # Coordinates for all pixels vs. the selected subset
        y_all, x_all = np.meshgrid(np.arange(self.height), np.arange(self.width), indexing='ij')
        omega_coords = np.column_stack([y_all.ravel(), x_all.ravel()])
        D_coords = np.column_stack([self.y_coords, self.x_coords])

        gamma_D = self.grayscale_image[self.y_coords, self.x_coords].astype(float)
        gamma_Omega = self.grayscale_image.ravel().astype(float)

        # 7) Compute kernel matrices
        K_D = kernel_function(D_coords, D_coords, gamma_D, gamma_D, sigma_1, sigma_2, p, gaussian_rbf)
        K_Omega = kernel_function(omega_coords, D_coords, gamma_Omega, gamma_D, sigma_1, sigma_2, p, gaussian_rbf)

        # 8) Regularize and solve
        print("Solving for interpolation coefficients...")
        n = len(self.y_coords)
        # Fix 5: Use the delta value from the slider instead of hardcoded value
        regularized_K_D = K_D + delta * n * np.eye(n)

        a = np.linalg.solve(regularized_K_D, self.f)
        F = (K_Omega @ a).reshape(self.height, self.width, 3)
        # 9) Interpolate for all pixels
        self.recolourised_image = F 

        # 10) Global mean scaling
        mean_orig = self.original_colours.mean(axis=(0, 1))
        mean_recolourised = self.recolourised_image.mean(axis=(0, 1))

        # Avoid division by zero
        mean_recolourised[mean_recolourised == 0] = 1e-8

        self.final_image = self.recolourised_image * (mean_orig / mean_recolourised)
        self.final_image = np.clip(self.final_image, 0, 255).astype(np.uint8)

        # 11) Save the final recoloured image
        self.final_path = ".\\final.jpeg" # should be improved
        print(f"Saving final recoloured image to: {self.final_path}")
        io.imsave(self.final_path, self.final_image)

        self.spinner.stop()  # Stop animation
        self.spinner_label.hide()  # Hide spinner

        self.final_widget = QLabel()
        pixmap = QPixmap(self.final_path)
        self.final_widget.setPixmap(pixmap)
        max_width, max_height = 400, 400
        self.final_widget.setAlignment(Qt.AlignmentFlag.AlignCenter)
        self.final_widget.setPixmap(pixmap.scaled(max_width, max_height, Qt.AspectRatioMode.KeepAspectRatio, Qt.TransformationMode.SmoothTransformation))
        self.main_layout.addWidget(self.final_widget, 2, 2)
        self.recolourize_button.setEnabled(True)


# Object class for popout selection window
class SelectionWindow(QDialog):

    def __init__(self, outer_instance):
        super().__init__()

        # Pass MainWindow as an argument
        self.outer = outer_instance

        # Name window and set size
        self.setWindowTitle("Sample Selector")
        self.setMinimumSize(QSize(500, 400))

        # Set topline text, similar to in MainWindow
        self.title_1 = QLabel("Select your sample!")
        self.title_1.setSizePolicy(QSizePolicy.Policy.Preferred, QSizePolicy.Policy.Fixed)  # Fix height

        # Create the canvas that the user will paint on. See SelectionCanvas class below
        self.label = SelectionCanvas(self, self.outer.grayscale_path, self.outer.image_path, self.outer.sampled_path)

        # Create row of buttons at bottom of screen
        self.ok_button = QDialogButtonBox.StandardButton.Ok
        self.reset_button = QDialogButtonBox.StandardButton.Reset
        self.cancel_button = QDialogButtonBox.StandardButton.Cancel
        buttons = (self.ok_button | self.reset_button | self.cancel_button)
        self.buttonBox = QDialogButtonBox(buttons)

        # Equip buttons wtih functions on click
        self.buttonBox.accepted.connect(self.on_ok_clicked)
        self.buttonBox.rejected.connect(self.on_cancel_clicked)
        self.buttonBox.clicked.connect(self.on_reset_clicked)

        # Create layout for selection window
        self.layout_1 = QVBoxLayout()
        self.layout_1.addWidget(self.title_1, alignment=Qt.AlignmentFlag.AlignCenter)
        self.layout_1.addWidget(self.label, alignment=Qt.AlignmentFlag.AlignCenter)
        self.layout_1.addWidget(self.buttonBox, alignment= Qt.AlignmentFlag.AlignBottom)
        self.layout_1.addStretch(1)

        # Add layout
        self.setLayout(self.layout_1)

    # Save sampled image and close selection window
    def on_ok_clicked(self):
        self.label.saveImage()
        self.outer.x_coords = np.array(self.outer.x_coords)
        self.outer.y_coords = np.array(self.outer.y_coords)
        self.accept()

    # Cancel out of selection window
    def on_cancel_clicked(self):
        self.reject()

    # Reset canvas to baseline
    def on_reset_clicked(self, button):
        if self.buttonBox.standardButton(button) == QDialogButtonBox.StandardButton.Reset:
            self.label.reset()

# Object class for the canvas users will paint on
class SelectionCanvas(QLabel):

    def __init__(self, outer_instance, base_layer, color_layer, save_path):
        super().__init__()
        
        # Pass SelectionWindow and file paths for grayscale, full_color and save location
        self.outer = outer_instance
        self.base_layer = base_layer
        self.color_layer = color_layer
        self.sampled_path = save_path

        # Set grayscale image as canvas
        self.reset()

        # Create hidden twin of full color image to use as paint palette
        self.color_label = QLabel()
        self.color_image = QPixmap(self.color_layer)
        self.color_label.setPixmap(self.color_image)
        #self.color_label.setPixmap(self.color_image.scaled(self.size(), Qt.AspectRatioMode.KeepAspectRatio, Qt.TransformationMode.SmoothTransformation))


    # Pull appropriate color at xy location from full color image
    def get_pen_color(self, event):
        x, y = int(event.position().x()), int(event.position().y())        
        return self.color_label.pixmap().toImage().pixelColor(x, y)
    
    # Set grayscale image as canvas
    def reset(self):
        self.grayscale_image = QImage(self.base_layer)
        pixmap = QPixmap(self.base_layer)
        self.setPixmap(pixmap)
        #self.setPixmap(self.pixmap().scaled(self.size(), Qt.AspectRatioMode.KeepAspectRatio, Qt.TransformationMode.SmoothTransformation))
        self.outer.outer.x_coords = []
        self.outer.outer.y_coords = []


    # When the user left-clicks (or holds down and moves mouse), set pen color to corresponding
    #   pixel color on full color image and paint pixel of that color. Update canvas with 
    #   corresponding changes.
    def mouseMoveEvent(self, event):
        x, y = int(event.position().x()), int(event.position().y())        

        canvas = self.pixmap()
        painter = QPainter(canvas)
        pen_color = self.get_pen_color(event)
        painter.setPen(pen_color)
        painter.drawPoint(x, y)
        if(x >= 0 and x < self.outer.outer.width and y >=0 and y < self.outer.outer.height):
            self.outer.outer.x_coords.append(x)
            self.outer.outer.y_coords.append(y)

        painter.end()

        self.setPixmap(canvas)
        self.update()

    # Save sampled image when "Ok" is clicked
    def saveImage(self):
        print(f"Saving sampled image to: {self.sampled_path}")
        self.pixmap().save(self.sampled_path, "JPEG")

if __name__ == "__main__":
    app = QApplication(sys.argv)
    window = MainWindow()
    window.show()
    app.exec()